# **RAG Pipeline**

In [ ]:
import pandas as pd
import json

# ========== Load sample ==========
sample_path = "arabicaqa_rag_results/dataset/df_sample_1000.csv"
df_sample_loaded = pd.read_csv(sample_path)

# ========== Fix list columns after loading ==========
list_columns = ["answers"]

for col in list_columns:
    df_sample_loaded[col] = df_sample_loaded[col].apply(
        lambda x: json.loads(x) if isinstance(x, str) and x.strip().startswith("[") else []
    )

print("✨ Loaded sample shape:", df_sample_loaded.shape)
df_sample_loaded.head()


✨ Loaded sample shape: (1000, 7)


,split,document_id,question_id,question,context,answers,is_impossible
0,train,1732461,1193607,متي تم بناء الموقع الأول لشركة توب غولف؟,توب غولف هي شركة ترفيهية رياضية عالمية مقرها ف...,[عام 2000],False
1,test,1583712,1160774,كم عدد الحفريات التي تم اكتشافها لببر نغاندونغ؟,ببر نغاندونغ هو نويع منقرض من أنواع الببور الح...,[سبع حفريات],False
2,train,1718678,1164879,ما هو مركز اللاعب ميو تساكتاش؟,ميو تساكتاش (8 مايو 1992 في سبليت في كرواتيا -...,[كصانع ألعاب],False
3,test,1582958,1068610,ما هي بعض المنتجات التي يمكن صنعها من القماش ا...,القماش الهَسِّيّ هو قماش منسوج يصنع عادة من أل...,[لصنع الحبال والشبكات والمنتجات المماثلة],False
4,test,1720179,1175188,ما هو مركز اللاعب رادو سابو؟,رادو سابو هو لاعب كرة قدم روماني في مركز الوسط...,[الوسط],False


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
# from langchain_community.llms import Ollama
from langchain_classic.chains import RetrievalQA


print("✅ all imports OK (new LangChain)")


c:\Users\Zohoor Almalki\Projects\NLP\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ all imports OK (new LangChain)


In [ ]:
# ========= 1) corpus with document_id =========
unique_contexts = (
    df_sample_loaded[["document_id", "context"]]
    .dropna(subset=["context"])
    .drop_duplicates(subset=["document_id"])
    .reset_index(drop=True)
)

len(unique_contexts), unique_contexts.iloc[0]["context"][:200]


(919,
 'توب غولف هي شركة ترفيهية رياضية عالمية مقرها في دالاس، تكساس ولها مواقع في الولايات المتحدة والمملكة المتحدة وأستراليا والمكسيك ودبي. تم الاستحواذ على الشركة من قبل Callaway Golf المتداولة علنًا في ما')

In [ ]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)

unique_contexts = (
    df_sample_loaded[["document_id", "context"]]
    .dropna(subset=["context"])
    .drop_duplicates(subset=["document_id"])
    .reset_index(drop=True)
)

docs = []
for _, row in unique_contexts.iterrows():
    doc_id = int(row["document_id"])
    text = row["context"]

    chunks = text_splitter.split_text(text)
    for i, chunk in enumerate(chunks):
        docs.append(
            Document(
                page_content=chunk,
                metadata={"document_id": doc_id, "chunk_id": i}
            )
        )

print("Example metadata:", docs[0].metadata)
print(f"✅ Created {len(docs)} chunks")


Example metadata: {'document_id': 1732461, 'chunk_id': 0}
✅ Created 6026 chunks


In [ ]:
import os
import gc
import time

import chromadb
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from tqdm.auto import tqdm

# ============================================================
# Chroma configuration
# ============================================================
CHROMA_DIR = "chroma_arabicaqa_docid"
COLLECTION_NAME = "arabicaqa_mrc_v1_docid"

# ============================================================
# Release old in-memory references if they exist
# ============================================================
for var_name in ["retriever", "vectordb", "vectorstore", "db", "client", "collection"]:
    if var_name in globals():
        try:
            del globals()[var_name]
            print(f"Deleted old variable reference: {var_name}")
        except Exception as e:
            print(f"Could not delete variable reference {var_name}: {e}")

gc.collect()
time.sleep(1)

# ============================================================
# Reset the Chroma collection without deleting the directory
# ============================================================
# This keeps the same CHROMA_DIR path while removing old persisted chunks
# from the target collection. It avoids Windows file-lock errors caused by
# deleting the full directory while Chroma files are still locked.

os.makedirs(CHROMA_DIR, exist_ok=True)

client = chromadb.PersistentClient(path=CHROMA_DIR)

try:
    client.delete_collection(COLLECTION_NAME)
    print(f"Deleted old Chroma collection: {COLLECTION_NAME}")
except Exception as e:
    print(f"No existing collection deleted, or collection was not found: {e}")

gc.collect()
time.sleep(1)

# ============================================================
# Create embeddings
# ============================================================
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

# ============================================================
# Recreate Chroma vector store in the same directory
# ============================================================
vectordb = Chroma(
    collection_name=COLLECTION_NAME,
    embedding_function=embeddings,
    persist_directory=CHROMA_DIR,
    client=client,
)

# ============================================================
# Add documents using stable unique IDs
# ============================================================
BATCH_SIZE = 5000

for i in tqdm(range(0, len(docs), BATCH_SIZE)):
    batch_docs = docs[i : i + BATCH_SIZE]

    # Stable unique IDs prevent duplicated documents inside Chroma.
    batch_ids = [
        f"doc_{d.metadata['document_id']}_chunk_{d.metadata['chunk_id']}"
        for d in batch_docs
    ]

    vectordb.add_documents(batch_docs, ids=batch_ids)

# ============================================================
# Persist the vector store and create the retriever
# ============================================================
vectordb.persist()

retriever = vectordb.as_retriever(search_kwargs={"k": 5})

print("✅ Chroma collection rebuilt without duplicated persisted chunks.")
print(f"✅ Chroma directory used: {CHROMA_DIR}")
print(f"✅ Chroma collection used: {COLLECTION_NAME}")

# ============================================================
# Verify that Chroma was rebuilt without duplicated chunks
# ============================================================

source_count = len(docs)
chroma_count = vectordb._collection.count()

print("Number of source docs:", source_count)
print("Number of Chroma docs:", chroma_count)

assert chroma_count == source_count, (
    f"Chroma count mismatch. Expected {source_count}, but found {chroma_count}. "
    "This may indicate duplicated or missing chunks."
)

print("✅ Verified: Chroma contains exactly one entry per source chunk.")


Deleted old Chroma collection: arabicaqa_mrc_v1_docid


C:\Users\Zohoor Almalki\AppData\Local\Temp\ipykernel_9336\2446374920.py:53: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
C:\Users\Zohoor Almalki\AppData\Local\Temp\ipykernel_9336\2446374920.py:60: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectordb = Chroma(
100%|██████████| 2/2 [03:59<00:00, 119.91s/it]

✅ Chroma collection rebuilt without duplicated persisted chunks.
✅ Chroma directory used: chroma_arabicaqa_docid
✅ Chroma collection used: arabicaqa_mrc_v1_docid
Number of source docs: 6026
Number of Chroma docs: 6026
✅ Verified: Chroma contains exactly one entry per source chunk.



C:\Users\Zohoor Almalki\AppData\Local\Temp\ipykernel_9336\2446374920.py:86: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  vectordb.persist()


In [ ]:
def deduplicate_docs_by_content_and_id(docs):
    seen = set()
    unique_docs = []

    for d in docs:
        key = (
            d.metadata.get("document_id"),
            d.metadata.get("chunk_id"),
            d.page_content.strip()
        )

        if key not in seen:
            unique_docs.append(d)
            seen.add(key)

    return unique_docs

question = df_sample_loaded.iloc[0]["question"]

retrieved_docs = retriever.invoke(question)
retrieved_docs = deduplicate_docs_by_content_and_id(retrieved_docs)

retrieved_contexts = [d.page_content for d in retrieved_docs]
retrieved_doc_ids = [d.metadata.get("document_id") for d in retrieved_docs]
retrieved_chunk_ids = [d.metadata.get("chunk_id") for d in retrieved_docs]

print(question)
print(retrieved_doc_ids)
print(retrieved_chunk_ids)
print(retrieved_contexts[0][:300])

متي تم بناء الموقع الأول لشركة توب غولف؟
[1732461, 1732461, 1727247, 1364429, 1583150]
[4, 5, 1, 5, 3]
في المملكة المتحدة. بعد ستة أشهر، كان لدى منشأة دالاس انتظار لمدة ست ساعات. استحوذت شركة توب غولف أنترناشيونال (الولايات المتحدة) على توب غولف (المملكة البريطانية) في عام 2009. ومن ثم في يوليو استحوذت على ورلد غولف سيستيمس وبالتحديد الجوانب الملكية الفكرية. وفي عام 2011 أصبح إريك أندرسون الرئيس التن


In [ ]:
q = df_sample_loaded.iloc[0]["question"]

retrieved_docs = retriever.invoke(q)
for i, d in enumerate(retrieved_docs):
    print(i)
    print(d.metadata)
    print(d.page_content[:150])
    print("-" * 80)

0
{'chunk_id': 4, 'document_id': 1732461}
في المملكة المتحدة. بعد ستة أشهر، كان لدى منشأة دالاس انتظار لمدة ست ساعات. استحوذت شركة توب غولف أنترناشيونال (الولايات المتحدة) على توب غولف (المملك
--------------------------------------------------------------------------------
1
{'document_id': 1732461, 'chunk_id': 5}
توب غولف إلى 163 مليون دولار. بحلول عام 2015، كان لدى توب غولف 28 موقعاً وجلبت لهم 8 ملايين زائر. اشترت توب غولف شركة «ورلد جلف تور» للألعاب في 2016. 
--------------------------------------------------------------------------------
2
{'chunk_id': 1, 'document_id': 1727247}
إلى 70 متجرًا في المملكة المتحدة، وتَقررأن العلامة التجارية بحاجة إلى هوية جديدة. تمت إعادة تسمية لويس باسم تشيلسي جيرل، ويعود اختيارهذا الاسم في ذلك 
--------------------------------------------------------------------------------
3
{'chunk_id': 5, 'document_id': 1364429}
تعُود الفكرة الرئيسيَّة وراء إعادة تنّظيم المعلومات حول مواقع الويب إلى عام 1995، عندما طور ومُطورون آخرون من مختبر أبحاث كمبيوتر أبل إط

## **Answers Generation**

In [ ]:
from dotenv import load_dotenv
load_dotenv()

True

In [ ]:
import os
import json
import time
import ast
import math
import requests
import pandas as pd
from tqdm import tqdm
from dotenv import load_dotenv

from typing import Optional, List, Any
from langchain_core.language_models.llms import LLM
from langchain_core.prompts import PromptTemplate
from langchain_core.callbacks.manager import CallbackManagerForLLMRun
from langchain_classic.chains import RetrievalQA

# ============================================================
# 0) Configuration
# ============================================================
TEST_MODE = False
N_ANSWERABLE_TEST = 1
N_UNANSWERABLE_TEST = 1

SKIP_IF_MODEL_FILE_EXISTS = False
MERGE_AT_END = True

SAMPLE_PATH = "arabicaqa_rag_results/dataset/df_sample_1000.csv"
OUTPUT_DIR = "arabicaqa_rag_results/predictions"
os.makedirs(OUTPUT_DIR, exist_ok=True)

FINAL_OUTPUT_CSV = os.path.join(OUTPUT_DIR, "comparison_llama_mistral_command_1000.csv")
FINAL_OUTPUT_JSON = os.path.join(OUTPUT_DIR, "comparison_llama_mistral_command_1000.json")

# ============================================================
# OpenRouter Configuration
# ============================================================
load_dotenv()

OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY")

if not OPENROUTER_API_KEY:
    raise ValueError("OPENROUTER_API_KEY environment variable is not set.")

OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1/chat/completions"

# ============================================================
# Models: OpenRouter model IDs
# ============================================================
MODELS = {
    "command": "cohere/command-r7b-12-2024",
    "llama":   "meta-llama/llama-3-8b-instruct", 
    "mistral": "mistralai/mistral-7b-instruct-v0.1" 
    # This version "mistralai/mistral-7b-instruct-v0.1" on OpenRouter will Going away May 30, 2026 
    # We used Ollama = mistral:7b-instruct, ID = 6577803aa9a0 in the first experement
  
}

# Free-tier safety delay to reduce rate-limit errors.
SLEEP_BETWEEN_REQUESTS = 0.0  # use 3.0 only for OpenRouter free-tier models

# ============================================================
# Custom LangChain LLM wrapper for OpenRouter
# ============================================================
class OpenRouterLLM(LLM):
    model_name: str
    api_key: str
    temperature: float = 0.0
    max_tokens: int = 64
    timeout: int = 120

    @property
    def _llm_type(self) -> str:
        return "openrouter"

    @property
    def _identifying_params(self):
        return {"model_name": self.model_name}

    def _call(
        self,
        prompt: str,
        stop: Optional[List[str]] = None,
        run_manager: Optional[CallbackManagerForLLMRun] = None,
        **kwargs: Any,
    ) -> str:
        headers = {
            "Authorization": f"Bearer {self.api_key}",
            "Content-Type": "application/json",
            "HTTP-Referer": "https://arabicaqa-rag.local",
            "X-Title": "ArabicaQA RAG Evaluation",
        }

        payload = {
            "model": self.model_name,
            "messages": [
                {
                    "role": "user",
                    "content": prompt,
                }
            ],
            "temperature": self.temperature,
            "max_tokens": self.max_tokens,
        }

        if stop:
            payload["stop"] = stop

        last_error = None

        for attempt in range(3):
            try:
                response = requests.post(
                    OPENROUTER_BASE_URL,
                    headers=headers,
                    json=payload,
                    timeout=self.timeout,
                )

                if response.status_code == 429:
                    wait = 60 * (attempt + 1)
                    print(f"\n[Rate limit hit] Waiting {wait}s before retry {attempt + 1}/3...")
                    time.sleep(wait)
                    continue

                response.raise_for_status()
                data = response.json()

                return data["choices"][0]["message"]["content"].strip()

            except requests.exceptions.Timeout as e:
                last_error = e
                print(f"\n[Timeout] Attempt {attempt + 1}/3 for model {self.model_name}")
                time.sleep(10)

            except requests.exceptions.RequestException as e:
                last_error = e
                print(f"\n[Request error] Attempt {attempt + 1}/3 for model {self.model_name}: {e}")
                time.sleep(10)

            except Exception as e:
                last_error = e
                print(f"\n[Unexpected error] Attempt {attempt + 1}/3 for model {self.model_name}: {e}")
                time.sleep(10)

        raise RuntimeError(
            f"OpenRouter call failed after 3 attempts for model {self.model_name}. "
            f"Last error: {last_error}"
        )

# ============================================================
# 1) Load the fixed evaluation subset
# ============================================================
df_sample = pd.read_csv(SAMPLE_PATH)

print("=" * 90)
print("SEQUENTIAL MULTI-MODEL RAG GENERATION - OpenRouter")
print("=" * 90)
print("Loaded sample:", SAMPLE_PATH)
print("Original shape:", df_sample.shape)

# ============================================================
# 2) Optional Smoke Test
# is_impossible == False => answerable
# is_impossible == True  => unanswerable
# ============================================================
if TEST_MODE:
    df_ans = df_sample[df_sample["is_impossible"] == False].head(N_ANSWERABLE_TEST)
    df_un = df_sample[df_sample["is_impossible"] == True].head(N_UNANSWERABLE_TEST)
    df_sample = pd.concat([df_ans, df_un], ignore_index=True)

    print("\nTEST MODE ENABLED")
    print("Smoke test shape:", df_sample.shape)
    print(df_sample["is_impossible"].value_counts())
else:
    print("\nFULL RUN MODE")
    print("Running all rows:", df_sample.shape)

# ============================================================
# 3) Basic validation and parsing
# ============================================================
required_cols = ["is_impossible", "question", "answers"]
missing = [c for c in required_cols if c not in df_sample.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

def parse_answers(x):
    if x is None:
        return []
    if isinstance(x, float) and math.isnan(x):
        return []
    if isinstance(x, list):
        return [str(v).strip() for v in x if str(v).strip()]
    if isinstance(x, str):
        s = x.strip()
        if not s:
            return []
        try:
            parsed = json.loads(s)
            if isinstance(parsed, list):
                return [str(v).strip() for v in parsed if str(v).strip()]
        except Exception:
            pass
        try:
            parsed = ast.literal_eval(s)
            if isinstance(parsed, list):
                return [str(v).strip() for v in parsed if str(v).strip()]
            return [str(parsed).strip()] if str(parsed).strip() else []
        except Exception:
            return [s]
    return []

df_sample["answers"] = df_sample["answers"].apply(parse_answers)
df_sample["is_impossible"] = df_sample["is_impossible"].astype(bool)
df_sample["is_answerable"] = ~df_sample["is_impossible"]

print("\nSample ready:", df_sample.shape)

# ============================================================
# 4) Shared prompt
# ============================================================
template = """
أجب عن السؤال التالي باللغة العربية الفصحى فقط،
وبأقصر وأدق إجابة ممكنة.
لا تضف أي شرح أو مقدمة أو معلومات من خارج السياق.
إذا لم تكن الإجابة موجودة في السياق، اكتب فقط: "غير موجود في السياق."

السؤال: {question}

السياق:
{context}

الإجابة:
"""

prompt = PromptTemplate(
    template=template,
    input_variables=["question", "context"]
)

# ============================================================
# 5) Helper: run one model fully
# ============================================================
def run_single_model(model_key, openrouter_model_name, df_eval):
    output_csv = os.path.join(OUTPUT_DIR, f"predictions_{model_key}_1000.csv")
    output_json = os.path.join(OUTPUT_DIR, f"predictions_{model_key}_1000.json")

    if SKIP_IF_MODEL_FILE_EXISTS and os.path.exists(output_csv):
        print("\n" + "=" * 90)
        print(f"SKIPPING {model_key}: existing file found")
        print(output_csv)
        print("=" * 90)
        return pd.read_csv(output_csv, encoding="utf-8-sig")

    print("\n" + "=" * 90)
    print(f"RUNNING MODEL: {model_key} -> {openrouter_model_name}")
    print("=" * 90)

    llm = OpenRouterLLM(
        model_name=openrouter_model_name,
        api_key=OPENROUTER_API_KEY,
        temperature=0.0,
        max_tokens=128,
        timeout=120,
    )

    qa_chain = RetrievalQA.from_chain_type(
        llm=llm,
        retriever=retriever,  # assumes retriever exists
        chain_type="stuff",
        chain_type_kwargs={"prompt": prompt},
        return_source_documents=True,
    )

    answer_col = f"predicted_answer_{model_key}"
    latency_col = f"latency_sec_{model_key}"
    error_col = f"error_{model_key}"

    results = []

    for idx, row in tqdm(
        df_eval.iterrows(),
        total=len(df_eval),
        desc=f"Generating with {model_key}"
    ):
        question = row["question"]

        record = {
            "row_idx": idx,
            "split": row.get("split", ""),
            "document_id": row.get("document_id", ""),
            "question_id": row.get("question_id", ""),
            "question": question,
            "context": row.get("context", ""),
            "correct_answers": row["answers"],
            "is_impossible": bool(row.get("is_impossible", False)),
            "is_answerable": not bool(row.get("is_impossible", False)),

            "retrieved_contexts": [],
            "retrieved_doc_ids": [],
            "retrieved_chunk_ids": [],

            answer_col: "ERROR",
            latency_col: None,
            error_col: "",
        }

        start_time = time.perf_counter()

        try:
            output = qa_chain.invoke({"query": question})

            source_docs = output.get("source_documents", [])

            record["retrieved_contexts"] = [
                d.page_content for d in source_docs
            ]

            record["retrieved_doc_ids"] = [
                d.metadata.get("document_id", None)
                for d in source_docs
            ]

            record["retrieved_chunk_ids"] = [
                d.metadata.get("chunk_id", None)
                for d in source_docs
            ]

            if isinstance(output, dict) and "result" in output:
                record[answer_col] = str(output["result"]).strip()
            else:
                record[answer_col] = str(output).strip()

        except Exception as e:
            record[error_col] = str(e)
            print(f"\n{model_key} error on question {idx}: {e}")

        end_time = time.perf_counter()
        record[latency_col] = end_time - start_time

        results.append(record)

        # OpenRouter free-tier safety delay
        time.sleep(SLEEP_BETWEEN_REQUESTS)

    df_model = pd.DataFrame(results)

    df_model.to_csv(output_csv, index=False, encoding="utf-8-sig")
    df_model.to_json(output_json, orient="records", lines=True, force_ascii=False)

    print("\nDONE:", model_key)
    print("Saved CSV :", output_csv)
    print("Saved JSON:", output_json)
    print("Rows:", len(df_model))

    n_errors = int((df_model[answer_col] == "ERROR").sum())
    mean_latency = df_model[latency_col].mean()
    median_latency = df_model[latency_col].median()
    p95_latency = df_model[latency_col].quantile(0.95)

    print("Errors:", n_errors)
    print(f"Mean latency:   {mean_latency:.3f} sec/query")
    print(f"Median latency: {median_latency:.3f} sec/query")
    print(f"P95 latency:    {p95_latency:.3f} sec/query")

    return df_model

# ============================================================
# 6) Run models sequentially
# ============================================================
model_dfs = {}

for model_key, openrouter_model_name in MODELS.items():
    df_model = run_single_model(model_key, openrouter_model_name, df_sample)
    model_dfs[model_key] = df_model

print("\n" + "=" * 90)
print("ALL REQUESTED MODEL RUNS FINISHED OR SKIPPED")
print("=" * 90)

# ============================================================
# 7) Merge model outputs
# ============================================================
if MERGE_AT_END:
    print("\nMerging model files...")

    model_files = {
        model_key: os.path.join(OUTPUT_DIR, f"predictions_{model_key}_1000.csv")
        for model_key in MODELS.keys()
    }

    existing_model_files = {
        model_key: path
        for model_key, path in model_files.items()
        if os.path.exists(path)
    }

    if not existing_model_files:
        raise FileNotFoundError("No model prediction files found to merge.")

    print("Files found:")
    for model_key, path in existing_model_files.items():
        print(f" - {model_key}: {path}")

    dfs = {}
    for model_key, path in existing_model_files.items():
        df = pd.read_csv(path, encoding="utf-8-sig")
        dfs[model_key] = df
        print(model_key, df.shape)

    # Prefer llama as base if available; otherwise first available model.
    base_model = "llama" if "llama" in dfs else list(dfs.keys())[0]
    print("\nBase model for metadata:", base_model)

    base_cols = [
        "row_idx",
        "split",
        "document_id",
        "question_id",
        "question",
        "context",
        "correct_answers",
        "is_impossible",
        "is_answerable",
        "retrieved_contexts",
        "retrieved_doc_ids",
        "retrieved_chunk_ids",
    ]

    available_base_cols = [c for c in base_cols if c in dfs[base_model].columns]

    df_merged = dfs[base_model][available_base_cols].copy()

    for model_key, df_model in dfs.items():
        model_cols = [
            "row_idx",
            f"predicted_answer_{model_key}",
            f"latency_sec_{model_key}",
            f"error_{model_key}",
        ]

        missing_model_cols = [c for c in model_cols if c not in df_model.columns]
        if missing_model_cols:
            raise KeyError(f"Missing columns in {model_key} file: {missing_model_cols}")

        df_merged = df_merged.merge(
            df_model[model_cols],
            on="row_idx",
            how="left",
            validate="one_to_one",
        )

    df_merged = df_merged.sort_values("row_idx").reset_index(drop=True)

    df_merged.to_csv(FINAL_OUTPUT_CSV, index=False, encoding="utf-8-sig")
    df_merged.to_json(FINAL_OUTPUT_JSON, orient="records", lines=True, force_ascii=False)

    print("\nMerged results saved:")
    print("CSV :", FINAL_OUTPUT_CSV)
    print("JSON:", FINAL_OUTPUT_JSON)
    print("Shape:", df_merged.shape)

    print("\nMerged summary:")
    for model_key in existing_model_files.keys():
        answer_col = f"predicted_answer_{model_key}"
        latency_col = f"latency_sec_{model_key}"

        n_errors = int((df_merged[answer_col] == "ERROR").sum())
        mean_latency = df_merged[latency_col].mean()
        median_latency = df_merged[latency_col].median()
        p95_latency = df_merged[latency_col].quantile(0.95)

        print(f"\nModel: {model_key}")
        print(f"Errors:         {n_errors}")
        print(f"Mean latency:   {mean_latency:.3f} sec/query")
        print(f"Median latency: {median_latency:.3f} sec/query")
        print(f"P95 latency:    {p95_latency:.3f} sec/query")

    print("\nAnswerability counts:")
    print(df_merged["is_impossible"].value_counts())

    print("\nRetrieved evidence check:")
    print("retrieved_contexts missing:", df_merged["retrieved_contexts"].isna().sum())
    print("retrieved_doc_ids missing:", df_merged["retrieved_doc_ids"].isna().sum())
    print("retrieved_chunk_ids missing:", df_merged["retrieved_chunk_ids"].isna().sum())

    display(df_merged.head())

SEQUENTIAL MULTI-MODEL RAG GENERATION - OpenRouter
Loaded sample: arabicaqa_rag_results/dataset/df_sample_1000.csv
Original shape: (1000, 7)

FULL RUN MODE
Running all rows: (1000, 7)

Sample ready: (1000, 8)

RUNNING MODEL: command -> cohere/command-r7b-12-2024


Generating with command:   0%|          | 0/1000 [00:00<?, ?it/s]

Generating with command: 100%|██████████| 1000/1000 [25:14<00:00,  1.51s/it]



DONE: command
Saved CSV : arabicaqa_rag_results/predictions\predictions_command_1000.csv
Saved JSON: arabicaqa_rag_results/predictions\predictions_command_1000.json
Rows: 1000
Errors: 0
Mean latency:   1.512 sec/query
Median latency: 1.340 sec/query
P95 latency:    2.546 sec/query

RUNNING MODEL: llama -> meta-llama/llama-3-8b-instruct


Generating with llama: 100%|██████████| 1000/1000 [20:35<00:00,  1.24s/it]



DONE: llama
Saved CSV : arabicaqa_rag_results/predictions\predictions_llama_1000.csv
Saved JSON: arabicaqa_rag_results/predictions\predictions_llama_1000.json
Rows: 1000
Errors: 0
Mean latency:   1.233 sec/query
Median latency: 1.122 sec/query
P95 latency:    1.892 sec/query

RUNNING MODEL: mistral -> mistralai/mistral-7b-instruct-v0.1


Generating with mistral: 100%|██████████| 1000/1000 [1:47:28<00:00,  6.45s/it]



DONE: mistral
Saved CSV : arabicaqa_rag_results/predictions\predictions_mistral_1000.csv
Saved JSON: arabicaqa_rag_results/predictions\predictions_mistral_1000.json
Rows: 1000
Errors: 0
Mean latency:   6.445 sec/query
Median latency: 6.019 sec/query
P95 latency:    10.231 sec/query

ALL REQUESTED MODEL RUNS FINISHED OR SKIPPED

Merging model files...
Files found:
 - command: arabicaqa_rag_results/predictions\predictions_command_1000.csv
 - llama: arabicaqa_rag_results/predictions\predictions_llama_1000.csv
 - mistral: arabicaqa_rag_results/predictions\predictions_mistral_1000.csv
command (1000, 15)
llama (1000, 15)
mistral (1000, 15)

Base model for metadata: llama

Merged results saved:
CSV : arabicaqa_rag_results/predictions\comparison_llama_mistral_command_1000.csv
JSON: arabicaqa_rag_results/predictions\comparison_llama_mistral_command_1000.json
Shape: (1000, 21)

Merged summary:

Model: command
Errors:         0
Mean latency:   1.512 sec/query
Median latency: 1.340 sec/query
P95 

,row_idx,split,document_id,question_id,question,context,correct_answers,is_impossible,is_answerable,retrieved_contexts,...,retrieved_chunk_ids,predicted_answer_command,latency_sec_command,error_command,predicted_answer_llama,latency_sec_llama,error_llama,predicted_answer_mistral,latency_sec_mistral,error_mistral
0,0,train,1732461,1193607,متي تم بناء الموقع الأول لشركة توب غولف؟,توب غولف هي شركة ترفيهية رياضية عالمية مقرها ف...,['عام 2000'],False,True,['في المملكة المتحدة. بعد ستة أشهر، كان لدى من...,...,"[4, 5, 1, 5, 3]",غير موجود في السياق.,3.673629,NaN,1995.,1.433770,NaN,غير موجود في السياق للسنة التي تم بناء الموقع ...,5.209762,NaN
1,1,test,1583712,1160774,كم عدد الحفريات التي تم اكتشافها لببر نغاندونغ؟,ببر نغاندونغ هو نويع منقرض من أنواع الببور الح...,['سبع حفريات'],False,True,['ولما أزيح التراب من فوق هرم. اكتشفت 21 مقبرة...,...,"[1, 0, 2, 4, 5]",تم اكتشاف سبع حفريات لببر نغاندونغ.,1.807783,NaN,21,1.845930,NaN,غير موجود في السياق للعدد الخاص بالحفريات التي...,5.265048,NaN
2,2,train,1718678,1164879,ما هو مركز اللاعب ميو تساكتاش؟,ميو تساكتاش (8 مايو 1992 في سبليت في كرواتيا -...,['كصانع ألعاب'],False,True,"['بورت غالب.', 'ما بعد لافوازييه.', 'البنية.',...",...,"[8, 8, 76, 4, 0]",غير موجود في السياق.,1.694605,NaN,لاعب وسط.,1.491944,NaN,غير موجود في السياق لمعرفة مركز ميو تساكتاش.,4.074893,NaN
3,3,test,1582958,1068610,ما هي بعض المنتجات التي يمكن صنعها من القماش ا...,القماش الهَسِّيّ هو قماش منسوج يصنع عادة من أل...,['لصنع الحبال والشبكات والمنتجات المماثلة'],False,True,['المعلومات الغذائية عن مقلوبة الباذنجان.\nتحت...,...,"[2, 1, 0, 0, 2]",غير موجود في السياق.,1.623586,NaN,القماش الهسي.,1.695586,NaN,غير موجود في السياق.,2.316705,NaN
4,4,test,1720179,1175188,ما هو مركز اللاعب رادو سابو؟,رادو سابو هو لاعب كرة قدم روماني في مركز الوسط...,['الوسط'],False,True,"['بورت غالب.', 'فترتي «استعراش كنمو» ثم «نان ب...",...,"[8, 29, 8, 0, 0]",غير موجود في السياق.,1.568155,NaN,لا يوجد في السياق.,1.107627,NaN,غير موجود في السياق لمعرفة مركز لاعب رادو سابو.,4.023189,NaN


In [ ]:
# ============================================================
# 0) Paths + Load df_results
# ============================================================
import os
import pandas as pd

output_dir = "arabicaqa_rag_results/predictions"
df_results_name = "comparison_llama_mistral_command_1000.csv"
df_results_path = os.path.join(output_dir, df_results_name)

os.makedirs(output_dir, exist_ok=True)

if not os.path.exists(df_results_path):
    raise FileNotFoundError(f"df_results not found at: {df_results_path}")

df_results = pd.read_csv(df_results_path)
df_results.head()

,row_idx,split,document_id,question_id,question,context,correct_answers,is_impossible,is_answerable,retrieved_contexts,...,retrieved_chunk_ids,predicted_answer_command,latency_sec_command,error_command,predicted_answer_llama,latency_sec_llama,error_llama,predicted_answer_mistral,latency_sec_mistral,error_mistral
0,0,train,1732461,1193607,متي تم بناء الموقع الأول لشركة توب غولف؟,توب غولف هي شركة ترفيهية رياضية عالمية مقرها ف...,['عام 2000'],False,True,['في المملكة المتحدة. بعد ستة أشهر، كان لدى من...,...,"[4, 5, 1, 5, 3]",غير موجود في السياق.,3.673629,NaN,1995.,1.433770,NaN,غير موجود في السياق للسنة التي تم بناء الموقع ...,5.209762,NaN
1,1,test,1583712,1160774,كم عدد الحفريات التي تم اكتشافها لببر نغاندونغ؟,ببر نغاندونغ هو نويع منقرض من أنواع الببور الح...,['سبع حفريات'],False,True,['ولما أزيح التراب من فوق هرم. اكتشفت 21 مقبرة...,...,"[1, 0, 2, 4, 5]",تم اكتشاف سبع حفريات لببر نغاندونغ.,1.807783,NaN,21,1.845930,NaN,غير موجود في السياق للعدد الخاص بالحفريات التي...,5.265048,NaN
2,2,train,1718678,1164879,ما هو مركز اللاعب ميو تساكتاش؟,ميو تساكتاش (8 مايو 1992 في سبليت في كرواتيا -...,['كصانع ألعاب'],False,True,"['بورت غالب.', 'ما بعد لافوازييه.', 'البنية.',...",...,"[8, 8, 76, 4, 0]",غير موجود في السياق.,1.694605,NaN,لاعب وسط.,1.491944,NaN,غير موجود في السياق لمعرفة مركز ميو تساكتاش.,4.074893,NaN
3,3,test,1582958,1068610,ما هي بعض المنتجات التي يمكن صنعها من القماش ا...,القماش الهَسِّيّ هو قماش منسوج يصنع عادة من أل...,['لصنع الحبال والشبكات والمنتجات المماثلة'],False,True,['المعلومات الغذائية عن مقلوبة الباذنجان.\nتحت...,...,"[2, 1, 0, 0, 2]",غير موجود في السياق.,1.623586,NaN,القماش الهسي.,1.695586,NaN,غير موجود في السياق.,2.316705,NaN
4,4,test,1720179,1175188,ما هو مركز اللاعب رادو سابو؟,رادو سابو هو لاعب كرة قدم روماني في مركز الوسط...,['الوسط'],False,True,"['بورت غالب.', 'فترتي «استعراش كنمو» ثم «نان ب...",...,"[8, 29, 8, 0, 0]",غير موجود في السياق.,1.568155,NaN,لا يوجد في السياق.,1.107627,NaN,غير موجود في السياق لمعرفة مركز لاعب رادو سابو.,4.023189,NaN


In [ ]:
# ============================================================
# Data Integrity Check:
# Empty gold answers vs. is_impossible labels
# ============================================================

import os
import math
import pandas as pd

def is_empty_gold(x):
    if x is None:
        return True

    if isinstance(x, float) and math.isnan(x):
        return True

    if isinstance(x, (list, tuple)):
        cleaned = [str(t).strip() for t in x if str(t).strip()]
        return len(cleaned) == 0

    return str(x).strip() == ""

empty_mask = df_results["correct_answers"].apply(is_empty_gold)

# Make sure is_impossible is boolean
df_results["is_impossible"] = df_results["is_impossible"].astype(bool)

empty_gold_count = int(empty_mask.sum())
empty_gold_pct = round(empty_mask.mean() * 100, 2)

empty_gold_but_answerable = df_results[
    empty_mask & (df_results["is_impossible"] == False)
].copy()

nonempty_gold_but_unanswerable = df_results[
    (~empty_mask) & (df_results["is_impossible"] == True)
].copy()

print("=" * 80)
print("DATA INTEGRITY CHECK: GOLD ANSWERS vs ANSWERABILITY")
print("=" * 80)
print(f"Total rows: {len(df_results):,}")
print(f"Empty gold count: {empty_gold_count:,}")
print(f"Empty gold %: {empty_gold_pct:.2f}%")
print("-" * 80)
print(
    "Empty gold but is_impossible=False count:",
    len(empty_gold_but_answerable)
)
print(
    "Non-empty gold but is_impossible=True count:",
    len(nonempty_gold_but_unanswerable)
)
print("=" * 80)

# Save diagnostics
diagnostics_dir = "arabicaqa_rag_results/diagnostics"
os.makedirs(diagnostics_dir, exist_ok=True)

summary = pd.DataFrame([
    {
        "total_rows": len(df_results),
        "empty_gold_count": empty_gold_count,
        "empty_gold_percent": empty_gold_pct,
        "empty_gold_but_answerable_count": len(empty_gold_but_answerable),
        "nonempty_gold_but_unanswerable_count": len(nonempty_gold_but_unanswerable),
    }
])

summary_path = os.path.join(diagnostics_dir, "gold_answer_integrity_summary.csv")
summary.to_csv(summary_path, index=False, encoding="utf-8-sig")

if len(empty_gold_but_answerable) > 0:
    path1 = os.path.join(diagnostics_dir, "empty_gold_but_answerable_rows.csv")
    empty_gold_but_answerable.to_csv(path1, index=False, encoding="utf-8-sig")
    print("Saved problematic rows:", path1)

if len(nonempty_gold_but_unanswerable) > 0:
    path2 = os.path.join(diagnostics_dir, "nonempty_gold_but_unanswerable_rows.csv")
    nonempty_gold_but_unanswerable.to_csv(path2, index=False, encoding="utf-8-sig")
    print("Saved inconsistent rows:", path2)

print("Saved summary:", summary_path)

summary

DATA INTEGRITY CHECK: GOLD ANSWERS vs ANSWERABILITY
Total rows: 1,000
Empty gold count: 0
Empty gold %: 0.00%
--------------------------------------------------------------------------------
Empty gold but is_impossible=False count: 0
Non-empty gold but is_impossible=True count: 500
Saved inconsistent rows: arabicaqa_rag_results/diagnostics\nonempty_gold_but_unanswerable_rows.csv
Saved summary: arabicaqa_rag_results/diagnostics\gold_answer_integrity_summary.csv


,total_rows,empty_gold_count,empty_gold_percent,empty_gold_but_answerable_count,nonempty_gold_but_unanswerable_count
0,1000,0,0.0,0,500


# **Context-Window Overflow Diagnostic for the Legacy Mistral-7B Endpoint**

In [1]:
"""
Diagnostic for Reviewer #1, Concern #3
======================================
Measures how many evaluation prompts exceeded the OpenRouter context window of
the legacy `mistralai/mistral-7b-instruct-v0.1` endpoint, and tests whether
prompt length is associated with Mistral's over-abstention on answerable
questions.

Context on why this is needed
-----------------------------
The generation notebook posts to OpenRouter without a `transforms` or `plugins`
field. OpenRouter enables context compression BY DEFAULT for every endpoint
with <= 8192 tokens of context. The v0.1 endpoint is well under that bound
(OpenRouter lists 4096; third-party trackers report 2824 served). So any prompt
exceeding the window was silently compressed from the MIDDLE and returned a
normal 200 response. Nothing in the saved predictions records this.

Command-R7B (128k) and LLaMA 3 8B (8192, prompts ~2k) never triggered it.

Requires network on first run to fetch the tokenizer:
    pip install transformers pandas numpy
"""

import ast
import json
import re
import numpy as np
import pandas as pd

# ----------------------------------------------------------------------
# Configuration -- adjust paths to match your repo layout
# ----------------------------------------------------------------------
PRED_JSON = "arabicaqa_rag_results/predictions/comparison_llama_mistral_command_1000.json"
OUT_DIR = "arabicaqa_rag_results/diagnostics"

# max_tokens passed to OpenRouterLLM in the generation run
MAX_COMPLETION_TOKENS = 128

# Report against both candidate window sizes; the conclusion should be stated
# against the conservative (smaller) one.
CANDIDATE_WINDOWS = {"openrouter_page_4096": 4096, "served_2824": 2824}

# EXACT template from the generation notebook. Do not substitute the version
# printed in the paper appendix -- they differ (see note in the writeup).
TEMPLATE = """
أجب عن السؤال التالي باللغة العربية الفصحى فقط،
وبأقصر وأدق إجابة ممكنة.
لا تضف أي شرح أو مقدمة أو معلومات من خارج السياق.
إذا لم تكن الإجابة موجودة في السياق، اكتب فقط: "غير موجود في السياق."

السؤال: {question}

السياق:
{context}

الإجابة:
"""


def coerce_list(x):
    """retrieved_contexts round-trips through CSV/JSON as a stringified list."""
    if isinstance(x, list):
        return x
    if isinstance(x, str) and x.strip().startswith("["):
        try:
            return ast.literal_eval(x)
        except Exception:
            return []
    return []


def build_prompt(question, contexts):
    """Reproduce LangChain RetrievalQA(chain_type='stuff') prompt assembly."""
    return TEMPLATE.format(question=question, context="\n\n".join(contexts))


# ----------------------------------------------------------------------
# 1) Load predictions and rebuild the exact prompts that were sent
# ----------------------------------------------------------------------
df = pd.read_json(PRED_JSON, orient="records", lines=True)
df["retrieved_contexts"] = df["retrieved_contexts"].apply(coerce_list)
df["is_impossible"] = df["is_impossible"].astype(bool)
df["prompt"] = [
    build_prompt(q, c) for q, c in zip(df["question"], df["retrieved_contexts"])
]
df["prompt_chars"] = df["prompt"].str.len()

# Sanity: did any request actually fail? If these are both zero, that is
# positive evidence that compression was active rather than erroring out.
n_error_sentinel = int((df["predicted_answer_mistral"] == "ERROR").sum())
n_error_msg = int(df["error_mistral"].fillna("").astype(str).str.strip().ne("").sum())
print(f"Mistral rows with ERROR sentinel : {n_error_sentinel}")
print(f"Mistral rows with error message  : {n_error_msg}")

# Duplicate retrieved chunks inflate prompt length. The notebook defines
# deduplicate_docs_by_content_and_id() but never applies it in the generation
# loop, so report how often duplicates occurred.
dup_rows = int(sum(len(c) != len(set(c)) for c in df["retrieved_contexts"]))
print(f"Rows with duplicate retrieved chunks: {dup_rows}")

# ----------------------------------------------------------------------
# 2) Tokenize with the real v0.1 tokenizer
# ----------------------------------------------------------------------
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained("mistralai/Mistral-7B-Instruct-v0.1")
df["prompt_tokens"] = [len(tok(p, add_special_tokens=True).input_ids)
                       for p in df["prompt"]]

# Arabic encodes very inefficiently under the v0.1 32k SentencePiece vocab
# (heavy byte fallback), so the tokens-per-character ratio is the headline
# number that invalidates any character-based reasoning about fit.
tpc = df["prompt_tokens"].sum() / df["prompt_chars"].sum()

print("\n--- Prompt length (tokens) ---")
print(df["prompt_tokens"].describe(percentiles=[0.5, 0.95, 0.99]).round(1))
print(f"tokens per character (Arabic, v0.1 vocab): {tpc:.3f}")

print("\n--- Overflow ---")
overflow_summary = {}
for label, window in CANDIDATE_WINDOWS.items():
    budget = window - MAX_COMPLETION_TOKENS
    over = df["prompt_tokens"] > budget
    overflow_summary[label] = {
        "window": window,
        "input_budget": budget,
        "n_over": int(over.sum()),
        "pct_over": round(100 * over.mean(), 2),
        "n_over_answerable": int((over & ~df["is_impossible"]).sum()),
        "n_over_unanswerable": int((over & df["is_impossible"]).sum()),
    }
    print(f"{label}: budget={budget}, over={over.sum()} "
          f"({100 * over.mean():.1f}%)")

# Conservative window drives the safe-subset definition
CONSERVATIVE = min(CANDIDATE_WINDOWS.values())
df["compressed"] = df["prompt_tokens"] > (CONSERVATIVE - MAX_COMPLETION_TOKENS)

# ----------------------------------------------------------------------
# 3) Dose-response: does over-abstention track prompt length?
# ----------------------------------------------------------------------
# If middle-out compression causes Mistral's 64.2% false-abstention rate, the
# rate should rise with prompt length. If it is flat -- including in the
# shortest quartile, where nothing could have been removed -- compression is
# not the mechanism, and that is the strongest available defence.
#
# Replace with your own matcher from the evaluation notebook so the definition
# is identical to the one used for the reported figures.
ABSTAIN_PATTERNS = [
    r"غير\s*موجود\s*في\s*السياق",
    r"لا\s*(يوجد|توجد|أستطيع|يمكن)",
    r"not\s+(found|present|mentioned|available)",
    r"no\s+answer",
]
ABSTAIN_RE = re.compile("|".join(ABSTAIN_PATTERNS), re.IGNORECASE)


def is_abstention(text):
    return bool(ABSTAIN_RE.search(str(text)))


ans = df[~df["is_impossible"]].copy()
ans["false_abstain"] = ans["predicted_answer_mistral"].apply(is_abstention)
ans["len_q"] = pd.qcut(ans["prompt_tokens"], 4,
                       labels=["Q1 (shortest)", "Q2", "Q3", "Q4 (longest)"])

print("\n--- Mistral false abstention by prompt-length quartile (answerable) ---")
by_q = ans.groupby("len_q", observed=True).agg(
    n=("false_abstain", "size"),
    rate=("false_abstain", "mean"),
    median_tokens=("prompt_tokens", "median"),
)
by_q["rate"] = (100 * by_q["rate"]).round(2)
print(by_q)

# Permutation test for a monotone length effect: point-biserial correlation
# between prompt length and false abstention.
rng = np.random.default_rng(42)
x = ans["prompt_tokens"].to_numpy(float)
y = ans["false_abstain"].to_numpy(float)
obs = np.corrcoef(x, y)[0, 1]
null = np.array([np.corrcoef(x, rng.permutation(y))[0, 1] for _ in range(10000)])
p_trend = float((np.abs(null) >= abs(obs)).mean())
print(f"\npoint-biserial r = {obs:+.4f}, permutation p = {p_trend:.4f} (B=10,000)")

# Same check restricted to prompts that provably fit: any residual association
# here cannot be caused by compression at all.
safe = ans[~ans["compressed"]]
if len(safe) > 20:
    print(f"\nFalse-abstention rate on provably-uncompressed answerable "
          f"prompts (n={len(safe)}): {100 * safe['false_abstain'].mean():.2f}%")

# ----------------------------------------------------------------------
# 4) Emit the truncation-safe subset for re-scoring all three models
# ----------------------------------------------------------------------
safe_ids = df.loc[~df["compressed"], "row_idx"].tolist()
print(f"\nTruncation-safe subset: {len(safe_ids)} / {len(df)} instances "
      f"({(~df['compressed'] & ~df['is_impossible']).sum()} answerable, "
      f"{(~df['compressed'] & df['is_impossible']).sum()} unanswerable)")

import os
os.makedirs(OUT_DIR, exist_ok=True)

df[["row_idx", "is_impossible", "prompt_chars", "prompt_tokens", "compressed"]] \
    .to_csv(f"{OUT_DIR}/mistral_prompt_lengths.csv", index=False)

with open(f"{OUT_DIR}/context_overflow_summary.json", "w") as f:
    json.dump({
        "max_completion_tokens": MAX_COMPLETION_TOKENS,
        "tokens_per_char_arabic": round(tpc, 4),
        "prompt_tokens": {
            "mean": float(df["prompt_tokens"].mean()),
            "median": float(df["prompt_tokens"].median()),
            "p95": float(df["prompt_tokens"].quantile(0.95)),
            "max": int(df["prompt_tokens"].max()),
        },
        "overflow": overflow_summary,
        "n_error_sentinel": n_error_sentinel,
        "n_error_message": n_error_msg,
        "rows_with_duplicate_chunks": dup_rows,
        "false_abstention_by_quartile": by_q.to_dict(),
        "trend_r": obs,
        "trend_p": p_trend,
        "safe_subset_size": len(safe_ids),
    }, f, indent=2, ensure_ascii=False)

print(f"\nWrote diagnostics to {OUT_DIR}/")
print("Next: re-score all three models on safe_ids and confirm the ordering.")

Mistral rows with ERROR sentinel : 0
Mistral rows with error message  : 0
Rows with duplicate retrieved chunks: 9


c:\Users\Zohoor Almalki\Projects\NLP\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\Zohoor Almalki\Projects\NLP\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Zohoor Almalki\.cache\huggingface\hub\models--mistralai--Mistral-7B-Instruct-v0.1. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an admi


--- Prompt length (tokens) ---
count    1000.0
mean     1867.4
std       495.7
min       254.0
50%      1988.0
95%      2453.0
99%      2518.0
max      2590.0
Name: prompt_tokens, dtype: float64
tokens per character (Arabic, v0.1 vocab): 0.888

--- Overflow ---
openrouter_page_4096: budget=3968, over=0 (0.0%)
served_2824: budget=2696, over=0 (0.0%)

--- Mistral false abstention by prompt-length quartile (answerable) ---
                 n   rate  median_tokens
len_q                                   
Q1 (shortest)  125  74.40         1283.0
Q2             126  65.08         1882.5
Q3             125  63.20         2171.0
Q4 (longest)   124  62.90         2398.5

point-biserial r = -0.1402, permutation p = 0.0011 (B=10,000)

False-abstention rate on provably-uncompressed answerable prompts (n=500): 66.40%

Truncation-safe subset: 1000 / 1000 instances (500 answerable, 500 unanswerable)

Wrote diagnostics to arabicaqa_rag_results/diagnostics/
Next: re-score all three models on safe_ids 